# 📊 GenWealth AI — Phase 3 End-to-End Demo
## `03_advisor_engine_demo.ipynb` — Step 3.4

**Purpose**: Full demonstration of the Phase 3 Multi-LLM Advisory Pipeline.

### Pipeline Architecture
```
Phase 1 CSV              Phase 2 PKL              MongoDB Atlas
(enriched_rl_data)  →   (rl_backtest_metrics) →  (financial_knowledge)
        │                       │                        │
        └───────────────────────┴────────────────────────┘
                                │
                    ContextAggregator.build_ticker_context()
                                │
              ┌─────────────────┴──────────────────┐
              │           Groq Stage 1              │
              │    Intent Classification             │
              │  (llama-3.3-70b-versatile)          │
              └─────────────────┬──────────────────┘
                                │
              ┌─────────────────┴──────────────────┐
              │         Gemini Stage 2              │
              │    Deep Financial Analysis          │
              │     (gemini-2.5-flash)              │
              └─────────────────┬──────────────────┘
                                │
              ┌─────────────────┴──────────────────┐
              │         Groq Stage 3               │
              │    Verification Critic              │
              │  (llama-3.3-70b-versatile)          │
              └─────────────────┬──────────────────┘
                                │
                     Guardrails + Disclaimer
                                │
                     Final Advisory Report
```

---
| Section | Description |
|---|---|
| **Cell 1** | Environment setup & imports |
| **Cell 2** | Vector store freshness test (24h gate) |
| **Cell 3** | Context aggregation for NVDA + RELIANCE.NS |
| **Cell 4** | Full `LLMAdvisorEngine` pipeline run |
| **Cell 5** | Guardrails inspection + final report display |

---
## Cell 1 — Environment Setup & Imports

In [ ]:
# ============================================================
# Cell 1: Environment Setup & Imports
# ============================================================
import os
import sys
import json
import pprint
from pathlib import Path
from datetime import datetime, timezone

# ── Ensure project root is on sys.path ──────────────────────
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

# ── Load .env ───────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv()
print(f"Notebook started at: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")

# ── Environment variable checks ─────────────────────────────
REQUIRED_VARS = ["MONGODB_URI", "GEMINI_API_KEY", "GROQ_API_KEY"]
missing = [v for v in REQUIRED_VARS if not os.getenv(v)]
if missing:
    raise EnvironmentError(
        f"Missing required environment variables: {missing}\n"
        "Ensure they are set in your .env file at the project root."
    )
print("✅ All required environment variables found:")
for v in REQUIRED_VARS:
    val = os.getenv(v)
    masked = val[:8] + "..." + val[-4:] if val and len(val) > 12 else "***"
    print(f"   {v:20s} = {masked}")

# ── Import Phase 3 modules ───────────────────────────────────
from src.advisor.vector_store    import (
    get_mongo_collection,
    get_live_stock_news,
    query_knowledge_base,
    FRESHNESS_HOURS,
    _make_utc_aware,        # Internal helper — exposed for demo transparency
)
from src.advisor.context_builder import (
    ContextAggregator,
    get_phase1_snapshot,
    get_phase2_allocation,
)
from src.advisor.llm_engine import (
    LLMAdvisorEngine,
    classify_query_intent,
    generate_gemini_report,
    verify_report_accuracy,
)
from src.advisor.guardrails import (
    sanitize_and_append_disclaimer,
    get_compliance_flags,
)

print("\n✅ All Phase 3 modules imported successfully.")
print(f"   FRESHNESS_HOURS gate = {FRESHNESS_HOURS} hours")

---
## Cell 2 — Vector Store Test with 24h Freshness Refresh

Demonstrates the 24-hour staleness detection logic:
- Check the age of the newest stored document for NVDA.
- If stale, `query_knowledge_base()` auto-fetches via DuckDuckGo and upserts.
- If fresh, it serves the cached result without a network call.

> **Timezone safety**: All comparisons use UTC-aware datetimes via `_make_utc_aware()`.

In [ ]:
# ============================================================
# Cell 2: Vector Store Freshness Test
# ============================================================
TICKER_TEST = "NVDA"
NOW_UTC = datetime.now(timezone.utc)

print("=" * 60)
print(f"  Vector Store Freshness Test — {TICKER_TEST}")
print("=" * 60)

# ── 2A: Inspect existing document age ───────────────────────
print("\n[2A] Inspecting stored document age for NVDA...")
col = get_mongo_collection()
doc_count = col.count_documents({"ticker": TICKER_TEST})
print(f"  Total stored docs for {TICKER_TEST}: {doc_count}")

if doc_count > 0:
    newest = col.find_one(
        {"ticker": TICKER_TEST},
        sort=[("timestamp", -1)],
        projection={"timestamp": 1, "text_content": 1, "_id": 0}
    )
    raw_ts = newest["timestamp"]
    utc_ts = _make_utc_aware(raw_ts)   # UTC-aware normalisation
    age_hrs = (NOW_UTC - utc_ts).total_seconds() / 3600.0
    print(f"  Newest document timestamp : {utc_ts.strftime('%Y-%m-%d %H:%M:%S UTC')}")
    print(f"  Age                       : {age_hrs:.2f} hours")
    print(f"  Freshness threshold       : {FRESHNESS_HOURS} hours")
    is_stale = age_hrs > FRESHNESS_HOURS
    print(f"  Status                    : {'⚠️  STALE — will refresh' if is_stale else '✅ FRESH — cache hit'}")
else:
    print(f"  No existing documents → Cold-start will trigger on first query.")

# ── 2B: Execute query (triggers freshness logic internally) ──
print("\n[2B] Executing query_knowledge_base() — watch for refresh or cache-hit log...")
results = query_knowledge_base(
    ticker=TICKER_TEST,
    query_text="NVIDIA GPU AI demand earnings revenue outlook",
    top_k=3,
)

print(f"\n  Returned {len(results)} document(s):")
for i, doc in enumerate(results, 1):
    ts = _make_utc_aware(doc['timestamp']) if doc.get('timestamp') else None
    ts_str = ts.strftime('%Y-%m-%d %H:%M UTC') if ts else 'N/A'
    print(f"  [{i}] Score: {doc['similarity_score']:.4f} | Stored: {ts_str}")
    print(f"       Source: {doc['metadata'].get('source', 'N/A')}")
    print(f"       Title : {doc['metadata'].get('title', 'N/A')[:80]}")
    print(f"       Snippet: {doc['text_content'][:100]}...\n")

---
## Cell 3 — Context Aggregation (NVDA + RELIANCE.NS)

Demonstrates `ContextAggregator` assembling the full tri-phase context for two tickers:
- **NVDA** → USD-denominated US stock
- **RELIANCE.NS** → INR-denominated NSE stock

The currency context is explicitly tagged to prevent cross-scale confusion in the LLM.

In [ ]:
# ============================================================
# Cell 3: Context Aggregation — NVDA + RELIANCE.NS
# ============================================================
agg = ContextAggregator()
TICKERS_TO_DEMO = ["NVDA", "RELIANCE.NS"]

contexts = {}
prompts  = {}

for ticker in TICKERS_TO_DEMO:
    print("=" * 60)
    print(f"  Context Aggregation — {ticker}")
    print("=" * 60)

    # ── Phase 1: Quant snapshot ───────────────────────────────
    print(f"\n[3A] Phase 1 Snapshot for '{ticker}'...")
    try:
        p1 = get_phase1_snapshot(ticker)
        print(f"  as_of_date    : {p1['as_of_date']}")
        print(f"  close_price   : {p1['close_price']:,.4f}")
        print(f"  phase1_signal : {p1['phase1_signal']:.4f}")
        print(f"  sentiment     : {p1['sentiment']:.4f}")
        print(f"  vol_ratio     : {p1['vol_ratio']:.4f}")
    except Exception as e:
        print(f"  ERROR: {e}")

    # ── Phase 2: RL portfolio state ───────────────────────────
    print(f"\n[3B] Phase 2 PPO Allocation...")
    try:
        p2 = get_phase2_allocation(ticker)
        print(f"  total_return    : {p2['total_return_pct']}%")
        print(f"  sharpe_ratio    : {p2['sharpe_ratio']}")
        print(f"  max_drawdown    : {p2['max_drawdown_pct']}%")
        print(f"  alpha_vs_ew     : {p2['alpha_vs_equal_weight']}")
        print(f"  allocation      : {p2['allocation']}")
    except Exception as e:
        print(f"  ERROR: {e}")

    # ── Full context build ────────────────────────────────────
    print(f"\n[3C] Building full LLM context for '{ticker}'...")
    ctx = agg.build_ticker_context(ticker)
    prompt = agg.build_llm_prompt_context(ticker)
    contexts[ticker] = ctx
    prompts[ticker]  = prompt

    print(f"  Context keys    : {list(ctx.keys())}")
    print(f"  RAG docs        : {len(ctx.get('phase3_rag', []))}")
    print(f"  Prompt length   : {len(prompt):,} chars")

    print()

# ── Print NVDA prompt preview ─────────────────────────────────
print("\n" + "=" * 60)
print("  LLM Prompt Preview — NVDA (first 1000 chars)")
print("=" * 60)
print(prompts["NVDA"][:1000])
print("...")

---
## Cell 4 — Full Multi-LLM Pipeline Execution

Runs the complete `LLMAdvisorEngine` for NVDA:

```
Stage 0: ContextAggregator  →  tri-phase context payload
Stage 1: Groq  (Llama-70b)  →  intent classification
Stage 2: Gemini (2.5-flash)  →  institutional-grade report (4 sections)
Stage 3: Groq  (Llama-70b)  →  numeric fact-checking critic
Stage 4: Guardrails          →  compliance scan + disclaimer
```

In [ ]:
# ============================================================
# Cell 4: Full LLMAdvisorEngine Pipeline — NVDA
# ============================================================
import time

PIPELINE_TICKER = "NVDA"
USER_QUERY = "Should I increase my NVDA position given the current AI market dynamics?"

print("=" * 60)
print("  LLMAdvisorEngine — Full Pipeline Run")
print(f"  Ticker : {PIPELINE_TICKER}")
print(f"  Query  : {USER_QUERY}")
print("=" * 60 + "\n")

engine = LLMAdvisorEngine()
t_start = time.time()

result = engine.run_advisory_pipeline(
    ticker=PIPELINE_TICKER,
    user_query=USER_QUERY,
)

elapsed = time.time() - t_start

print("\n" + "=" * 60)
print("  Pipeline Summary")
print("=" * 60)
print(f"  pipeline_status : {result['pipeline_status']}")
print(f"  ticker          : {result['ticker']}")
print(f"  currency        : {result['currency']} ({result['exchange']})")
print(f"  intent          : {result['intent']}")
print(f"  is_accurate     : {result['is_accurate']}")
print(f"  critic_flags    : {len(result['critic_flags'])} flag(s)")
if result['critic_flags']:
    for flag in result['critic_flags']:
        print(f"    ⚠️  {flag}")
print(f"  final_report    : {len(result['final_report']):,} chars")
print(f"  elapsed_time    : {elapsed:.1f}s")

print("\n" + "=" * 60)
print("  Raw Context Snapshot (Phase 1 Quant + Phase 2 RL)")
print("=" * 60)
p1_snap = result['raw_context'].get('phase1_quant', {})
p2_alloc = result['raw_context'].get('phase2_portfolio', {})
print(f"  Phase1 signal : {p1_snap.get('phase1_signal', 'N/A')}")
print(f"  Sentiment     : {p1_snap.get('sentiment', 'N/A')}")
print(f"  PPO return    : {p2_alloc.get('total_return_pct', 'N/A')}%")
print(f"  Sharpe ratio  : {p2_alloc.get('sharpe_ratio', 'N/A')}")

# Store result for Cell 5
pipeline_result = result
print("\n✅ Pipeline result stored as `pipeline_result` for Cell 5.")

---
## Cell 5 — Guardrails Inspection + Final Report

- Runs standalone guardrail scan with `get_compliance_flags()`
- Demonstrates `strict_mode=True` vs. standard output
- Renders the full, verified, compliant final report

In [ ]:
# ============================================================
# Cell 5: Guardrails Inspection + Final Report Display
# ============================================================
from IPython.display import Markdown, display

print("=" * 60)
print("  Guardrails Compliance Scan")
print("=" * 60)

gemini_raw = pipeline_result["gemini_report"]

# ── 5A: Standalone compliance scan ───────────────────────────
print("\n[5A] Scanning Gemini report for compliance violations...")
flags = get_compliance_flags(gemini_raw)
if not flags:
    print("  ✅ CLEAN — No compliance violations detected.")
else:
    print(f"  ⚠️  {len(flags)} compliance flag(s) detected:")
    for f in flags:
        print(f"     - {f}")

# ── 5B: Critic verification summary ──────────────────────────
print("\n[5B] Critic Verification Summary...")
print(f"  is_accurate : {pipeline_result['is_accurate']}")
critic_flags = pipeline_result["critic_flags"]
if not critic_flags:
    print("  ✅ Critic: No numeric discrepancies detected.")
else:
    print(f"  ⚠️  {len(critic_flags)} discrepancy flag(s):")
    for cf in critic_flags:
        print(f"     - {cf}")

# ── 5C: Demonstrate strict_mode guardrail on a test string ───
print("\n[5C] Strict Mode Demo — injecting banned phrase...")
SYNTHETIC_VIOLATION = (
    gemini_raw[:200]
    + "\n\nNote: This strategy offers guaranteed returns with zero risk.\n"
    + gemini_raw[200:400]
)
strict_output = sanitize_and_append_disclaimer(SYNTHETIC_VIOLATION, strict_mode=True)
has_alert = "COMPLIANCE ALERT" in strict_output
print(f"  Compliance alert injected : {'✅ YES' if has_alert else '❌ NO'}")
print(f"  Output length             : {len(strict_output):,} chars")

# ── 5D: Display the FINAL verified + compliant report ────────
print("\n" + "=" * 60)
print("  FINAL ADVISORY REPORT (Rendered Markdown)")
print("=" * 60)

final_report = pipeline_result["final_report"]

# Header card
header_md = f"""
---
### 🏦 GenWealth AI — Investment Advisory Report
**Ticker**: `{pipeline_result['ticker']}`  
**Currency**: {pipeline_result['currency']} ({pipeline_result['exchange']})  
**Intent Classified**: `{pipeline_result['intent']}`  
**Accuracy Verified**: {'✅ Accurate' if pipeline_result['is_accurate'] else '⚠️ Flags Raised' if pipeline_result['is_accurate'] is False else '🔍 Verification Pending'}  
**Pipeline Status**: `{pipeline_result['pipeline_status']}`  

---
"""

display(Markdown(header_md + final_report))

---
## Cell 6 (Bonus) — RELIANCE.NS Full Pipeline Run

Validates the INR/NSE currency tagging path and confirms the pipeline works for Indian market tickers.

In [ ]:
# ============================================================
# Cell 6 (Bonus): RELIANCE.NS Pipeline — INR Currency Path
# ============================================================
print("=" * 60)
print("  LLMAdvisorEngine — RELIANCE.NS (INR / NSE)")
print("=" * 60 + "\n")

reliance_result = engine.run_advisory_pipeline(
    ticker="RELIANCE.NS",
    user_query="What is the current investment outlook for Reliance Industries?",
)

print(f"  pipeline_status : {reliance_result['pipeline_status']}")
print(f"  ticker          : {reliance_result['ticker']}")
print(f"  currency        : {reliance_result['currency']} ({reliance_result['exchange']})")
print(f"  intent          : {reliance_result['intent']}")
print(f"  is_accurate     : {reliance_result['is_accurate']}")
print(f"  critic_flags    : {reliance_result['critic_flags']}")
print(f"  final_report    : {len(reliance_result['final_report']):,} chars")

print("\n" + "-" * 60)
print("  RELIANCE.NS Report Preview (first 800 chars):")
print("-" * 60)
print(reliance_result["final_report"][:800])
print("...")

print("\n✅ Phase 3 end-to-end demo complete.")
print("   All pipeline stages verified: Vector Store → Context → Groq → Gemini → Groq → Guardrails")